In [ ]:
!pip install -q transformers datasets accelerate scikit-learn flask


In [ ]:
#first of all we have created a directory and import all necessary libararies
CSV_PATH = "/content/master_clauses.csv"
OUT_DIR  = "/content/contract-analyzer"
MODEL_DIR = f"{OUT_DIR}/models/legal-bert-clf"
DATA_DIR  = f"{OUT_DIR}/data/clf"

import os, json, re, numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

Path(DATA_DIR).mkdir(parents=True, exist_ok=True)
df = pd.read_csv(CSV_PATH)

# here we find base clause columns that have a matching "-Answer"/"- Answer"
base_cols = []
for c in df.columns:
    if c.endswith("-Answer") or c.endswith("- Answer"):
        base = c.replace("-Answer", "").replace("- Answer", "").strip()
        if base in df.columns:
            base_cols.append(base)

# some column in base column are duplicate so we try to remove them
seen, label_names = set(), []
for b in base_cols:
    if b not in seen:
        label_names.append(b)
        seen.add(b)
#this collect the document name and all other clause if they are present
def row_to_text(row, clause_names):
    parts = []
    # add document name if present
    if "Document Name" in row and str(row["Document Name"]).strip():
        parts.append(f"Document Name: {str(row['Document Name']).strip()}")
    for b in clause_names:
        a_col = None
        for variant in (f"{b}-Answer", f"{b}- Answer"):
            if variant in df.columns:
                a_col = variant
                break
        if a_col is None:
            continue
        val = row.get(a_col, "")
        if isinstance(val, float) and np.isnan(val):
            val = ""
        val = str(val).strip()
        if val:
            parts.append(f"{b}: {val}")
    return "\n".join(parts).strip()
#it convert the row to labels like [1 0 0]
def row_to_labels(row, clause_names):
    y = []
    for b in clause_names:
        a_col = None
        for variant in (f"{b}-Answer", f"{b}- Answer"):
            if variant in df.columns:
                a_col = variant
                break
        val = "" if a_col is None else row.get(a_col, "")
        if isinstance(val, float) and np.isnan(val):
            val = ""
        y.append(1 if str(val).strip() else 0)
    return y
#getting data for training
records = []
for _, r in df.iterrows():
    text = row_to_text(r, label_names)
    labels = row_to_labels(r, label_names)
    if any(labels):
        records.append({"text": text, "labels": labels})

prep_df = pd.DataFrame(records)
train_df, val_df = train_test_split(prep_df, test_size=0.2, random_state=42, shuffle=True)
#saving the data so that it is in hugging face format
def write_jsonl(df, fp):
    with open(fp, "w", encoding="utf-8") as f:
        for _, row in df.iterrows():
            f.write(json.dumps({"text": row["text"], "labels": row["labels"]}, ensure_ascii=False) + "\n")

write_jsonl(train_df, f"{DATA_DIR}/train.jsonl")
write_jsonl(val_df,   f"{DATA_DIR}/val.jsonl")
with open(f"{DATA_DIR}/labels.txt", "w", encoding="utf-8") as f:
    for n in label_names:
        f.write(n + "\n")

len(label_names), len(train_df), len(val_df)


(41, 408, 102)

In [ ]:
import inspect, transformers, torch
from transformers import TrainingArguments

print("Transformers version:", transformers.__version__)

def make_args(output_dir, lr=3e-5, train_bs=16, eval_bs=16, epochs=5, logging_steps=50):

    base = dict(
        output_dir=output_dir,
        learning_rate=lr,
        per_device_train_batch_size=train_bs,
        per_device_eval_batch_size=eval_bs,
        num_train_epochs=epochs,
        logging_steps=logging_steps,
        fp16=torch.cuda.is_available(),
    )

    sig = inspect.signature(TrainingArguments.__init__)
    params = sig.parameters

    # Prefer epoch-based eval/save if supported
    if "evaluation_strategy" in params and "save_strategy" in params:
        base.update(
            evaluation_strategy="epoch",
            save_strategy="epoch",
        )
        if "load_best_model_at_end" in params:
            base["load_best_model_at_end"] = True
        if "metric_for_best_model" in params:
            base["metric_for_best_model"] = "macro_f1"
        if "greater_is_better" in params:
            base["greater_is_better"] = True
    else:
        # Older versions: use step-based knobs if present; remove unsupported ones.
        if "do_eval" in params:
            base["do_eval"] = True
        if "eval_steps" in params:
            base["eval_steps"] = 500
        if "save_steps" in params:
            base["save_steps"] = 500

        for k in ["evaluation_strategy", "save_strategy", "load_best_model_at_end",
                  "metric_for_best_model", "greater_is_better"]:
            base.pop(k, None)


    base = {k: v for k, v in base.items() if k in params}
    return TrainingArguments(**base)


Transformers version: 4.57.1


In [ ]:
import numpy as np, torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, TrainingArguments, Trainer
from torch import nn
from pathlib import Path

MODEL_NAME = "nlpaueb/legal-bert-base-uncased"  # Legal-BERT
Path(MODEL_DIR).mkdir(parents=True, exist_ok=True)

# load labels
with open(f"{DATA_DIR}/labels.txt", "r", encoding="utf-8") as f:
    label_names = [ln.strip() for ln in f if ln.strip()]
num_labels = len(label_names)

# load dataset
ds = load_dataset("json", data_files={"train": f"{DATA_DIR}/train.jsonl",
                                      "validation": f"{DATA_DIR}/val.jsonl"})

tok = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess(batch):
    enc = tok(batch["text"], truncation=True, padding="max_length", max_length=512)
    enc["labels"] = batch["labels"]
    return enc

ds_tok = ds.map(preprocess, batched=True, remove_columns=ds["train"].column_names)
ds_tok = ds_tok.with_format("torch")

class MultiLabelModel(nn.Module):
    def __init__(self, base_model_name, num_labels):
        super().__init__()
        self.base = AutoModel.from_pretrained(base_model_name)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.base.config.hidden_size, num_labels)
    def forward(self, input_ids=None, attention_mask=None, labels=None):
        out = self.base(input_ids=input_ids, attention_mask=attention_mask)
        pooled = out.last_hidden_state[:,0]  # CLS
        logits = self.classifier(self.dropout(pooled))
        loss = None
        if labels is not None:
            loss = nn.BCEWithLogitsLoss()(logits, labels.float())
        return {"loss": loss, "logits": logits}
#it convert raw logits to probability using sigmoid
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1/(1+np.exp(-logits))
    preds = (probs >= 0.5).astype(int)
    tp = (preds & labels).sum(axis=0)
    fp = (preds & (1-labels)).sum(axis=0)
    fn = ((1-preds) & labels).sum(axis=0)
    f1_per_label = np.where((2*tp + fp + fn) > 0, (2*tp) / (2*tp + fp + fn), 0.0)
    macro_f1 = float(f1_per_label.mean()) if len(f1_per_label) else 0.0
    TP, FP, FN = tp.sum(), fp.sum(), fn.sum()
    micro_f1 = float((2*TP) / (2*TP + FP + FN + 1e-8))
    subset_acc = float((preds == labels).all(axis=1).mean())
    return {"macro_f1": macro_f1, "micro_f1": micro_f1, "subset_acc": subset_acc}

model = MultiLabelModel(MODEL_NAME, num_labels)

args = make_args(
    output_dir=MODEL_DIR,
    lr=3e-5,
    train_bs=32,
    eval_bs=32,
    epochs=9,
    logging_steps=90,
)


trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"],
    tokenizer=tok,
    compute_metrics=compute_metrics
)

trainer.train()
trainer.save_model(MODEL_DIR)
tok.save_pretrained(MODEL_DIR)
with open(f"{MODEL_DIR}/labels.txt","w",encoding="utf-8") as f:
    f.write("\n".join(label_names))

print("Saved:", MODEL_DIR)


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: rohitsin5312 (rohitsin5312-netaji-subhas-university-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
90,0.240500


Saved: /content/contract-analyzer/models/legal-bert-clf


In [ ]:

metrics = trainer.evaluate()
print("Validation metrics:", metrics)



Validation metrics: {'eval_loss': 0.1645926684141159, 'eval_macro_f1': 0.9877293457631601, 'eval_micro_f1': 0.9919557566603923, 'eval_subset_acc': 0.49019607843137253, 'eval_runtime': 0.778, 'eval_samples_per_second': 131.106, 'eval_steps_per_second': 5.141, 'epoch': 9.0}


In [ ]:


import os, re, json
import torch, torch.nn as nn
import numpy as np
import pandas as pd
from typing import List, Tuple
from transformers import AutoTokenizer, AutoModel
from safetensors.torch import load_file as safe_load_file


MODEL_DIR   = "/content/contract-analyzer/models/legal-bert-clf/checkpoint-117"
BASE_MODEL  = "nlpaueb/legal-bert-base-uncased"
CSV_PATH    = "/content/master_clauses.csv"
DATA_LABELS = "/content/contract-analyzer/data/clf/labels.txt"
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"


labels_fp_model = os.path.join(MODEL_DIR, "labels.txt")
labels_fp = labels_fp_model if os.path.exists(labels_fp_model) else DATA_LABELS
with open(labels_fp, "r", encoding="utf-8") as f:
    LABELS = [ln.strip() for ln in f if ln.strip()]

# tokenizer
# see tokenizer from checkpoint. if missing files, fall back to bse model.
try:
    tok = AutoTokenizer.from_pretrained(MODEL_DIR)
except Exception:
    tok = AutoTokenizer.from_pretrained(BASE_MODEL)

#make model head as we have created during training
class MultiLabelModel(nn.Module):
    def __init__(self, base_model_name, num_labels):
        super().__init__()
        self.base = AutoModel.from_pretrained(base_model_name)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.base.config.hidden_size, num_labels)

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, **kwargs):
        out = self.base(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        pooled = out.last_hidden_state[:, 0]  # CLS
        logits = self.classifier(self.dropout(pooled))
        return logits

#  Load trained weights from model.safetensors
model = MultiLabelModel(BASE_MODEL, len(LABELS))
st_path_safe = os.path.join(MODEL_DIR, "model.safetensors")
st_path_bin  = os.path.join(MODEL_DIR, "pytorch_model.bin")

if os.path.exists(st_path_safe):
    state = safe_load_file(st_path_safe, device="cpu")
elif os.path.exists(st_path_bin):
    state = torch.load(st_path_bin, map_location="cpu")
else:
    raise FileNotFoundError(f"No model.safetensors or pytorch_model.bin found in {MODEL_DIR}")

missing, unexpected = model.load_state_dict(state, strict=False)
print("Loaded state dict. Missing keys:", missing)
print("Unexpected keys:", unexpected)

model.to(DEVICE).eval()

# predictor
def predict_clause_presence(text: str, threshold: float = 0.5) -> Tuple[List[Tuple[str, float]], List[Tuple[str, float]]]:
    enc = tok(text, truncation=True, padding="max_length", max_length=512, return_tensors="pt")
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    with torch.no_grad():
        logits = model(**enc)
        probs = torch.sigmoid(logits).squeeze(0).cpu().numpy()
    scored = sorted([(LABELS[i], float(probs[i])) for i in range(len(LABELS))], key=lambda x: -x[1])
    preds = [(l, p) for l, p in scored if p >= threshold]
    return scored, preds

def predict_batch(texts: List[str], threshold: float = 0.5):
    enc = tok(texts, truncation=True, padding="max_length", max_length=512, return_tensors="pt")
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    with torch.no_grad():
        logits = model(**enc)               # [B, L]
        probs = torch.sigmoid(logits).cpu().numpy()
    out = []
    for row in probs:
        scored = sorted([(LABELS[i], float(row[i])) for i in range(len(LABELS))], key=lambda x: -x[1])
        preds = [(l, p) for l, p in scored if p >= threshold]
        out.append((scored, preds))
    return out

# convert CSV ROW to TRAINING-STYLE TEXT
_df_cache = None
def _load_csv():
    global _df_cache
    if _df_cache is None:
        _df_cache = pd.read_csv(CSV_PATH)
    return _df_cache

def csv_row_to_text(row: pd.Series, df: pd.DataFrame) -> str:
    parts = []
    if "Document Name" in df.columns and str(row.get("Document Name", "")).strip():
        parts.append(f"Document Name: {str(row.get('Document Name')).strip()}")
    for b in LABELS:
        a_col = None
        for variant in (f"{b}-Answer", f"{b}- Answer"):
            if variant in df.columns:
                a_col = variant; break
        if a_col is None:
            continue
        val = row.get(a_col, "")
        if isinstance(val, float) and np.isnan(val): val = ""
        val = str(val).strip()
        if val:
            parts.append(f"{b}: {val}")
    return "\n".join(parts).strip() or " "

def predict_from_csv_row(idx: int, threshold: float = 0.5):
    df = _load_csv()
    row = df.iloc[idx]
    text = csv_row_to_text(row, df)
    return predict_clause_presence(text, threshold=threshold)

# RAW CONTRACT → ANSWERS-STYLE (HEURISTICS)
DEFAULT_KEYS = {
    "Governing Law": ["governing law", "laws of", "jurisdiction", "venue"],
    "Agreement Date": ["dated", "date of this agreement", "as of"],
    "Effective Date": ["effective date", "commencement date", "becomes effective"],
    "Expiration Date": ["expiration", "expire on", "term ends"],
    "Renewal Term": ["renew", "renewal", "auto-renew", "extend for"],
    "Termination": ["terminate", "termination", "material breach", "for cause"],
    "Confidentiality": ["confidential", "non-disclosure", "nda", "confidentiality"],
    "Indemnity": ["indemnify", "indemnification", "hold harmless"],
    "Limitation Of Liability": ["limitation of liability", "shall not exceed", "cap on liability", "aggregate liability"],
    "Assignment": ["assign", "assignment", "may not assign"],
    "Change Of Control": ["change of control", "merger", "acquisition"],
    "Payment": ["payment", "fees", "invoice", "payable"],
    "Audit Rights": ["audit", "audit rights", "inspect records"],
    "Warranty": ["warranty", "warranties", "warrants", "as is"],
    "Insurance": ["insurance", "insured", "certificate of insurance"],
    "Third Party Beneficiary": ["third party beneficiary"],
    "Notice": ["notice", "notices", "address for notice", "deliver notice"],
}
def _split_sents(text: str): return re.split(r'(?<=[.!?])\s+', text)

def synthesize_answers_style(text: str, keys=DEFAULT_KEYS) -> str:
    sents = _split_sents(text)
    best = {}
    for clause, kws in keys.items():
        best_sent, best_score = "", -1
        for s in sents:
            score = sum(1 for k in kws if k in s.lower())
            if score > best_score:
                best_score, best_sent = score, s.strip()
        if best_sent:
            best[clause] = best_sent
    return "\n".join([f"{k}: {v}" for k, v in best.items() if v]) or text

def predict_from_raw_contract(text: str, threshold: float = 0.5):
    answers_text = synthesize_answers_style(text)
    return predict_clause_presence(answers_text, threshold=threshold)

# JSON REPORT (for UI/API)
def make_report(text: str, threshold: float = 0.5, auto_extract: bool = True, top_k: int = 10):
    scored, preds = (predict_from_raw_contract(text, threshold)
                     if auto_extract else
                     predict_clause_presence(text, threshold))
    return {
        "threshold": threshold,
        "auto_extract": auto_extract,
        "top_scores": [{"label": l, "score": s} for l, s in scored[:top_k]],
        "predicted": [{"label": l, "score": s} for l, s in preds],
        "input_preview": text[:800]
    }

# test
print("\nDemo A: single text")
demo = """Document Name: SERVICES AGREEMENT
Parties: ABC LLC and XYZ Inc.
Governing Law: State of California
Cap On Liability: total fees paid over 12 months.
Termination: Either party may terminate for material breach with 30 days’ notice."""
s, p = predict_clause_presence(demo, 0.5)
print("Top-5:", s[:5]); print("Preds:", p)

print("\nDemo B: batch")
b = ["Governing Law: New York. Confidentiality: recipient must keep information confidential.",
     "Payment: Fees due within 30 days of invoice. Audit Rights: Vendor shall maintain records."]
res = predict_batch(b, 0.5)
print("Batch[0] Top-5:", res[0][0][:5])

print("\nDemo C: CSV row #0 (set CSV_PATH if needed)")
try:
    s0, p0 = predict_from_csv_row(0, 0.5)
    print("Row0 Top-5:", s0[:5])
except Exception as e:
    print("CSV demo skipped:", e)

print("\nDemo D: raw contract (auto-extract)")
raw = """This Agreement is entered into as of March 1, 2024. The governing law shall be the laws of the State of New York.
Either party may terminate this Agreement for material breach upon thirty (30) days’ written notice.
this aggrement is made by fobis .there si no dilutional in this document
The aggregate liability shall not exceed the fees paid in the preceding twelve (12) months."""
print(json.dumps(make_report(raw, 0.5, True), indent=2))
# ===================== END ============================================================


Loaded state dict. Missing keys: []
Unexpected keys: []

Demo A: single text
Top-5: [('Affiliate License-Licensor', 0.911810576915741), ('Affiliate License-Licensee', 0.9039892554283142), ('Anti-Assignment', 0.9022459387779236), ('Insurance', 0.8877650499343872), ('Non-Disparagement', 0.8850293159484863)]
Preds: [('Affiliate License-Licensor', 0.911810576915741), ('Affiliate License-Licensee', 0.9039892554283142), ('Anti-Assignment', 0.9022459387779236), ('Insurance', 0.8877650499343872), ('Non-Disparagement', 0.8850293159484863), ('Warranty Duration', 0.8850064277648926), ('Competitive Restriction Exception', 0.8696539402008057), ('Covenant Not To Sue', 0.8691961169242859), ('License Grant', 0.8682963848114014), ('Rofr/Rofo/Rofn', 0.8637253642082214), ('Non-Transferable License', 0.8629958629608154), ('Post-Termination Services', 0.8603371381759644), ('Source Code Escrow', 0.8602823615074158), ('Ip Ownership Assignment', 0.8567540049552917), ('Minimum Commitment', 0.8560658693313599),

In [ ]:
#  simple heuristics for raw contracts
import re
from typing import Dict, List, Tuple

# clause -> indicative keywords (lowercase)
DEFAULT_KEYS: Dict[str, List[str]] = {
    "Agreement Date": ["dated", "date of this agreement", "as of", "executed on"],
    "Effective Date": ["effective date", "commencement", "becomes effective"],
    "Expiration Date": ["expiration", "expire on", "term ends", "end date"],
    "Renewal Term": ["renew", "renewal", "auto-renew", "extend for", "additional term"],
    "Term": ["term shall", "initial term", "during the term"],
    "Termination": ["terminate", "termination", "for cause", "for convenience", "material breach"],
    "Governing Law": ["governing law", "laws of", "jurisdiction", "venue"],
    "Confidentiality": ["confidential", "non-disclosure", "confidentiality", "proprietary"],
    "Indemnity": ["indemnify", "indemnification", "hold harmless", "defend"],
    "Limitation Of Liability": ["limitation of liability", "shall not exceed", "cap on liability", "aggregate liability"],
    "Assignment": ["assign", "assignment", "may not assign", "no assignment"],
    "Change Of Control": ["change of control", "merger", "acquisition", "sale of substantially all assets"],
    "Payment": ["payment", "fees", "invoice", "payable", "due within"],
    "Audit Rights": ["audit", "inspect records", "books and records"],
    "Warranty": ["warranty", "warranties", "as is", "disclaims"],
    "Insurance": ["insurance", "insured", "certificate of insurance", "coverage"],
    "Third Party Beneficiary": ["third party beneficiary", "no third party beneficiaries"],
    "Notice": ["notice", "notices", "address for notice", "deliver notice", "written notice"],
}

# --- basic split; avoids extra deps ---
_SENT_SPLIT = re.compile(r'(?<=[.!?])\s+')
def split_sentences(text: str) -> List[str]:
    sents = [s.strip() for s in _SENT_SPLIT.split(text) if s.strip()]
    # keep long lines without punctuation from being merged forever
    if len(sents) <= 1 and "\n" in text:
        sents = [ln.strip() for ln in text.splitlines() if ln.strip()]
    return sents

def score_sentence(sent_lower: str, keywords: List[str]) -> int:
    return sum(1 for k in keywords if k in sent_lower)

def best_sentence_for_clause(text: str, keys: Dict[str, List[str]]) -> Dict[str, str]:
    sents = split_sentences(text)
    lowers = [s.lower() for s in sents]
    out: Dict[str, str] = {}
    for clause, kws in keys.items():
        best_idx, best_score = -1, -1
        for i, ls in enumerate(lowers):
            sc = score_sentence(ls, kws)
            if sc > best_score:
                best_idx, best_score = i, sc
        if best_idx >= 0 and best_score > 0:
            out[clause] = sents[best_idx]
    return out

# --- mini extractors (regex-based, very rough but helpful) ---
_RE_DATE = re.compile(r'\b(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|'
                      r'Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|Oct(?:ober)?|'
                      r'Nov(?:ember)?|Dec(?:ember)?)[\s\-.,]*\d{1,2},\s*\d{4}\b|\b\d{1,2}/\d{1,2}/\d{2,4}\b', re.I)
_RE_MONEY = re.compile(r'\$\s?\d[\d,]*(?:\.\d+)?|\b\d+\s?(?:USD|dollars)\b', re.I)
_RE_PERCENT = re.compile(r'\b\d{1,3}\s?%\b')
_RE_DAYS = re.compile(r'\b(\d{1,3})\s*(?:calendar\s+)?days?\b', re.I)
_RE_LAW = re.compile(r'\blaws? of ([A-Z][A-Za-z .]+)\b')
_RE_CAP = re.compile(r'(?:shall not exceed|cap(?:ped)? at|maximum (?:aggregate )?liability(?: is|:)?)\s*([$\d,%. ]+)', re.I)

def extract_first(pattern: re.Pattern, text: str) -> str:
    m = pattern.search(text)
    return m.group(0) if m else ""

def extract_group1(pattern: re.Pattern, text: str) -> str:
    m = pattern.search(text)
    return m.group(1).strip() if m and m.lastindex else ""

def enrich_with_entities(clause2sent: Dict[str, str]) -> Dict[str, Dict[str, str]]:
    enriched = {}
    for clause, sent in clause2sent.items():
        info = {"sentence": sent}
        if "Date" in clause:
            info["date"] = extract_first(_RE_DATE, sent)
        if clause in ("Payment", "Limitation Of Liability"):
            money = extract_first(_RE_MONEY, sent)
            perc  = extract_first(_RE_PERCENT, sent)
            if money: info["amount"] = money
            if perc:  info["percent"] = perc
            if clause == "Limitation Of Liability":
                cap = extract_group1(_RE_CAP, sent) or money or perc
                if cap: info["cap"] = cap
        if clause == "Termination":
            days = extract_group1(_RE_DAYS, sent)
            if days: info["notice_days"] = days
        if clause == "Governing Law":
            juris = extract_group1(_RE_LAW, sent)
            if juris: info["jurisdiction"] = juris
        if clause == "Notice":
            days = extract_group1(_RE_DAYS, sent)
            if days: info["notice_days"] = days
        enriched[clause] = info
    return enriched

def synthesize_answers_style(text: str, keys: Dict[str, List[str]] = DEFAULT_KEYS) -> str:
    """Return '<Clause>: <best sentence>' lines suitable for your classifier input."""
    best = best_sentence_for_clause(text, keys)
    lines = [f"{k}: {v}" for k, v in best.items()]
    return "\n".join(lines) if lines else text.strip()

def analyze_contract(text: str, keys: Dict[str, List[str]] = DEFAULT_KEYS) -> Dict:
    """Structured summary: answers-style text + per-clause sentence + simple extracted values."""
    best = best_sentence_for_clause(text, keys)
    enriched = enrich_with_entities(best)
    return {
        "answers_style": "\n".join(f"{k}: {v['sentence']}" for k, v in enriched.items()),
        "clauses": enriched
    }

# quick demo
demo = """
This Agreement is entered into as of March 1, 2024 (the "Effective Date").
The governing law shall be the laws of the State of New York, without regard to conflicts of laws.
Either party may terminate this Agreement for material breach upon thirty (30) days' written notice.
The aggregate liability shall not exceed the total fees paid in the preceding twelve (12) months or $100,000, whichever is less.
All notices shall be delivered by certified mail to the addresses below.
"""

print("Answers-style:\n", synthesize_answers_style(demo))
print("\nStructured:")
from pprint import pprint; pprint(analyze_contract(demo))


Answers-style:
 Agreement Date: This Agreement is entered into as of March 1, 2024 (the "Effective Date").
Effective Date: This Agreement is entered into as of March 1, 2024 (the "Effective Date").
Termination: Either party may terminate this Agreement for material breach upon thirty (30) days' written notice.
Governing Law: The governing law shall be the laws of the State of New York, without regard to conflicts of laws.
Limitation Of Liability: The aggregate liability shall not exceed the total fees paid in the preceding twelve (12) months or $100,000, whichever is less.
Payment: The aggregate liability shall not exceed the total fees paid in the preceding twelve (12) months or $100,000, whichever is less.
Notice: Either party may terminate this Agreement for material breach upon thirty (30) days' written notice.

Structured:
{'answers_style': 'Agreement Date: This Agreement is entered into as of March '
                  '1, 2024 (the "Effective Date").\n'
                  'Effecti

In [ ]:
!pip install -q gradio PyPDF2 python-docx

import gradio as gr, json, os, io
from PyPDF2 import PdfReader
from docx import Document

# helpers for file text extraction
def _read_pdf(fp):
    reader = PdfReader(fp)
    return "\n".join(p.extract_text() or "" for p in reader.pages)

def _read_docx(fp):
    doc = Document(fp)
    return "\n".join(p.text for p in doc.paragraphs if p.text.strip())

def _read_txt(fp):
    data = fp.read()
    try:
        return data.decode("utf-8")
    except UnicodeDecodeError:
        return data.decode("latin-1", errors="ignore")

def analyze_contract_ui(file, text, threshold, auto_extract):
    # prefer file if provided
    if file is not None:
        ext = os.path.splitext(file.name)[1].lower()
        with open(file.name, "rb") as f:
            if ext == ".pdf":
                text = _read_pdf(f)
            elif ext == ".docx":
                text = _read_docx(f)
            elif ext == ".txt":
                text = _read_txt(f)
            else:
                return ("Unsupported file. Use .pdf, .docx, or .txt.", "", "", "",)
    text = (text or "").strip()
    if not text:
        return ("No input provided.", "", "", "",)

    rep = make_report(text, threshold=float(threshold), auto_extract=bool(auto_extract), top_k=10)

    preds = "\n".join(f"{x['label']:35s} {x['score']:.3f}" for x in rep["predicted"]) or "(none)"
    top10 = "\n".join(f"{x['label']:35s} {x['score']:.3f}" for x in rep["top_scores"]) or "(none)"
    ans_used = rep.get("answers_style_used_for_ml", rep.get("input_preview",""))
    full_json = json.dumps(rep, indent=2)
    return preds, top10, ans_used, full_json

with gr.Blocks() as demo:
    gr.Markdown("## Contract Analyzer (Legal-BERT + heuristics)")
    with gr.Row():
        file = gr.File(label="Upload contract (.pdf / .docx / .txt)", file_types=[".pdf", ".docx", ".txt"])
        text = gr.Textbox(lines=10, label="Or paste contract text")
    th = gr.Slider(0.0, 1.0, value=0.5, step=0.05, label="Threshold")
    auto = gr.Checkbox(value=True, label="Auto-extract key sentences (heuristics)")
    btn = gr.Button("Analyze")

    out_preds = gr.Textbox(label="Predicted clauses (≥ threshold)")
    out_top   = gr.Textbox(label="Top 10 scores")
    out_ans   = gr.Textbox(label="Answers-style used for ML", lines=10)
    out_json  = gr.Textbox(label="Full JSON", lines=14)

    btn.click(analyze_contract_ui, [file, text, th, auto], [out_preds, out_top, out_ans, out_json])

demo.launch(share=True)  # prints a public URL you can open


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 14.2 MB/s eta 0:00:00
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7d652fee471254b62c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
